# Value + Quality (Sector-Neutral) — Strategy Research Notebook

This notebook asks a specific research question:

> Does a composite of earnings yield (value) and ROE (quality), demeaned within sector each month, earn a better risk-adjusted return than either leg alone on the point-in-time S&P 500 universe?

In plain English: cheap *and* profitable stocks should do better than cheap alone (which often loads on distressed cyclicals) or quality alone (which often loads on expensive defensives). Sector demeaning tries to remove the industry bet so we are ranking *relative* cheapness/quality inside each sector.

Example:

```text
It is month-end March 2024. Technology sector has 50 S&P names.

Within Tech:
  Stock A: earnings yield 8%, ROE 25%  -> high within Tech on both -> LONG candidate
  Stock B: earnings yield 1%, ROE 5%   -> low within Tech on both  -> SHORT candidate

A utility with earnings yield 6% might look "cheap" vs the whole market but
average inside Utilities — after sector demeaning it is near zero and does
not drive the long/short book.
```

**Limitation (disclose):** sector labels are *today's* FMP profile sector applied to all history (mild lookahead). There is no point-in-time sector history on FMP.

Computation is imported from `core/signals/sector_neutral.py` and `core/strategies/factor_runner.py` — nothing is re-implemented here.

## 1. Setup And Data Loading

Paths resolve from the repository root whether Jupyter started in the root or inside `notebooks/`. Missing files stop the notebook with a clear message.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "core").is_dir() else _cwd.parent
assert (REPO_ROOT / "core").is_dir(), f"Could not locate repo root from {_cwd}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FACTORS_PATH = REPO_ROOT / "data" / "factors" / "factors_fundamental.parquet"
PRICES_PATH = REPO_ROOT / "data" / "factors" / "prices.parquet"
SECTORS_PATH = REPO_ROOT / "data" / "sectors" / "sector_classifications.parquet"
ADV_PATH = REPO_ROOT / "data" / "factors" / "dollar_adv_21d.parquet"

for path in (FACTORS_PATH, PRICES_PATH, SECTORS_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Build fundamentals + sectors before running this notebook."
        )

fundamental_factors = pd.read_parquet(FACTORS_PATH)
prices = pd.read_parquet(PRICES_PATH)
sectors = pd.read_parquet(SECTORS_PATH).set_index("symbol")["sector"]
dollar_adv = pd.read_parquet(ADV_PATH) if ADV_PATH.exists() else None

print(f"fundamental factors: {fundamental_factors.shape}")
print(f"price panel: {prices.shape} tz={prices.index.tz}")
print(f"sector map: {sectors.nunique()} sectors, {len(sectors)} symbols")
print(f"dollar ADV loaded: {dollar_adv is not None}")

## 2. What Exactly Are We Observing?

- **Universe**: point-in-time S&P 500 membership (`sp500_universe_filter`).
- **Value leg**: `earnings_yield` = TTM net income / market cap (PIT after filing `acceptedDate`).
- **Quality leg**: `roe` = TTM net income / book equity (same PIT rule).
- **Composite**: equal-weight average of cross-sectional z-scores of the two legs. When sector-neutral, each leg is demeaned within sector *before* z-scoring.
- **Returns**: daily adj-close returns; net of ADV-scaled costs when the ADV panel is present (else flat 10 bps).
- **What we do not see**: earnings before filing date; historical sector membership (today's sector only); names missing from FMP price coverage (survivorship gap).

## 3. Per-Variable Audit

### `earnings_yield`
- **Source**: `data/factors/factors_fundamental.parquet` via `core/data/factors/fundamentals.py`.
- **Transformation**: TTM net income / market cap.
- **Missing-data handling**: NaN when either input missing; negative book names do not affect this column.
- **Publication lag**: join on next trading day after the later of income/balance-sheet `acceptedDate`.
- **Standardization timing**: cross-sectional z-score fit/applied on each rebalance date only (no look-ahead across dates).
- **Leakage risk**: low if PIT panel is used as-is.

### `roe`
- **Source**: same fundamentals panel.
- **Transformation**: TTM net income / book equity; NaN when book ≤ 0.
- **Missing-data handling**: NaN excluded from ranking.
- **Publication lag**: same as earnings yield.
- **Standardization timing**: same date-wise z-score.
- **Leakage risk**: low for filings; medium for sector demeaning (today's sector label).

### Sector label
- **Source**: `data/sectors/sector_classifications.parquet` (FMP `/profile`).
- **Transformation**: none — static map symbol → sector.
- **Publication lag**: none (current label).
- **Leakage risk**: medium — mild lookahead for sector-neutral ranks.

## 4. Transparency Block

> When the model says "today looks like high value-quality", that means: on this month-end date, after (optional) subtracting each name's sector mean earnings yield and ROE, we z-score both legs across the S&P 500 cross-section and average them. The long book is the top quintile of that composite; the short book is the bottom quintile. Labels are constructed ranks, not ground-truth "cheap" or "quality" companies.

## 5. Build Composite Factors And Backtest

In [ ]:
from core.backtest.portfolio import (
    calculate_portfolio_returns,
    create_signals_from_factor,
    sp500_universe_filter,
)
from core.metrics.performance import calculate_performance_metrics
from core.signals.sector_neutral import combine_value_quality

START = pd.Timestamp("2015-01-01", tz="America/New_York")
END = pd.Timestamp("2025-12-31", tz="America/New_York")

f = fundamental_factors.loc[
    (fundamental_factors.index.get_level_values("date") >= START)
    & (fundamental_factors.index.get_level_values("date") <= END)
].copy()
p = prices.loc[START:END]
adv = None
if dollar_adv is not None:
    adv = dollar_adv.reindex(index=p.index).reindex(columns=p.columns)

membership = sp500_universe_filter()

raw_composite = combine_value_quality(f["earnings_yield"], f["roe"], sector_neutral=False)
sn_composite = combine_value_quality(
    f["earnings_yield"], f["roe"], symbol_to_sector=sectors, sector_neutral=True
)

# Attach composites onto a factor frame the shared signal builder understands.
panel = f[["earnings_yield", "roe"]].copy()
panel["value_quality"] = raw_composite
panel["value_quality_sn"] = sn_composite

specs = {
    "earnings_yield": "earnings_yield",
    "roe": "roe",
    "value_quality": "value_quality",
    "value_quality_sn": "value_quality_sn",
}

results = {}
for label, col in specs.items():
    signals = create_signals_from_factor(
        panel, col, top_pct=0.2, bottom_pct=0.2, universe_filter=membership
    )
    port = calculate_portfolio_returns(
        signals, p, transaction_cost=0.001, dollar_adv=adv
    )
    metrics = calculate_performance_metrics(port["net_return"])
    results[label] = {"port": port, "metrics": metrics}
    print(
        f"{label:20s} sharpe={metrics['sharpe_ratio']:+.2f} "
        f"ann_ret={metrics['annualized_return']:+.2%} "
        f"max_dd={metrics['max_drawdown']:.2%}"
    )

## 6. Overlay / Sizing Disclosure

- **Positioning**: equal-weight long top 20% and short bottom 20% of the ranked factor (dollar-neutral).
- **Rebalance**: month-end (`ME`).
- **Signal lag**: default 1 trading day inside `create_signals_from_factor`.
- **Costs**: ADV-bucket schedule when `dollar_adv_21d.parquet` is present; otherwise flat 10 bps one-way.
- **No overlay**: exposure is fully invested long/short each month (no vol targeting, no regime gate).

## 7. Decision

Compare Sharpe and drawdowns of the four columns above.

- If `value_quality_sn` beats both single legs *and* the non-neutral composite, sector bets were hurting the raw composite — keep sector-neutral as the research default.
- If sector-neutral is worse, the industry tilt was part of the premium (common for value) — do not "fix" it away without an economic reason.
- Prefer 2015+ windows because of the disclosed FMP survivorship gap for historical S&P members.

Register a strategy only after this notebook's conclusion is stable across a second split (e.g. 2018–2025 holdout).

In [ ]:
# Equity curves (net wealth index, start = 1.0)
fig, ax = plt.subplots(figsize=(10, 4))
for label, payload in results.items():
    wealth = (1.0 + payload["port"]["net_return"]).cumprod()
    ax.plot(wealth.index, wealth.values, label=label, linewidth=1.2)
ax.set_title("Value / quality variants — net wealth (2015+)")
ax.set_ylabel("Wealth")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()